In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("rahul_transactions.csv")


In [3]:
print(type(df))

<class 'pandas.core.frame.DataFrame'>


In [4]:
print("shape",df.shape)
print("\ncolumns")
print(df.columns)

print("\n First 5 rows")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nTransaction types:")
print(df["Type"].value_counts())

print("\nUnique descriptions:", df["Description"].nunique())



shape (1328, 8)

columns
Index(['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode',
       'Ref'],
      dtype='object')

 First 5 rows
          Date   Time                       Description   Type  Amount  \
0   2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1    01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2    01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   
3   2024-01-01  14:07              UPI-AMAN-8934@OKAXIS  Debit   ₹1828   
4  01 Jan 2024  14:23                      BHIM-BLINKIT  Debit  270.00   

    Balance  Mode        Ref  
0  678275.0   UPI  TXN190872  
1  681007.0   UPI  TXN143064  
2  484728.0  NEFT  TXN246316  
3 -748745.0   UPI  TXN569226  
4  680737.0   UPI  TXN968962  

Data types:
Date            object
Time            object
Description     object
Type            object
Amount          object
Balance        float64
Mode            object
Ref             object
dtype: object

Trans

In [5]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Rows before:",before)
print("Rows after:",after)
print("Duplicates remvod:",before - after)

Rows before: 1328
Rows after: 1310
Duplicates remvod: 18


In [6]:
df["date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

In [7]:
print("Invalid dates:", df["date"].isna().sum())
print("Start date:", df["date"].min())
print("End date:", df["date"].max())

Invalid dates: 0
Start date: 2024-01-01 00:00:00
End date: 2024-06-30 00:00:00


In [8]:
df["amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "")
    .str.replace("Rs.", "")
    .str.replace(",", "")
    .str.strip()
)

df["amount"] = pd.to_numeric(
    df["amount"],
    errors="coerce"
)

In [9]:
print(df[["Amount", "amount"]].head(10))
print("Invalid amounts:", df["amount"].isna().sum())

    Amount   amount
0    ₹2462   2462.0
1    50.00     50.0
2   ₹84728  84728.0
3    ₹1828   1828.0
4   270.00    270.0
5  Rs. 625    625.0
6  Rs. 148    148.0
7     ₹482    482.0
8  Rs. 537    537.0
9     3956   3956.0
Invalid amounts: 0


In [10]:
df["type"] = df["Type"].str.lower()

df["type"] = df["type"].replace({
    "dr": "debit",
    "debit": "debit",
    "cr": "credit",
    "credit": "credit"
})

In [11]:
print(df["type"].value_counts())

type
debit     1304
credit       6
Name: count, dtype: int64


In [12]:
print("Rows:", len(df))
print("Invalid dates:", df["date"].isna().sum())
print("Invalid amounts:", df["amount"].isna().sum())

print("\nTransaction types:")
print(df["type"].value_counts())

print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())

Rows: 1310
Invalid dates: 0
Invalid amounts: 0

Transaction types:
type
debit     1304
credit       6
Name: count, dtype: int64

Date range:
2024-01-01 00:00:00 to 2024-06-30 00:00:00


In [13]:
print(df["Description"].unique())

['AMAZON SELLER SVCS' 'BHIM-BMTC' 'NEFT-TECHCRUSH LABS-SALARY MAY24'
 'UPI-AMAN-8934@OKAXIS' 'BHIM-BLINKIT' 'BHIM ZEPTO'
 'UPI-UBER-2426@HDFCBANK' 'POS SWIGGY BANGALORE' 'UPI-GROWWPAY@HDFCBANK'
 'OLA ELECTRIC' 'BMS MOVIE TICKETS' 'POS OLA-PRIME' 'SWIGGY-INSTAMART'
 'UPI-STARBUCKS@AXIS' 'UPI-THIRDWAVE@OKAXIS' 'ANI Technologies'
 'BMTC BUS PASS' 'POS TRUFFLES' 'FLIPKART INDIA' 'POS SWIGGY-RESTAURANT'
 'GROFERS INDIA P L' 'POS UBER BANGALORE' 'BANGALORE ELEC SUPPLY'
 'TWC INDIA' 'UPI-BESCOM-BILL@HDFCBANK' 'UPI-AMAN-0816@OKAXIS'
 'ROPPEN TRANSPORTATION' 'OLA CABS' 'POS ZOMATO' 'UPI-AMAZONPAY@HDFCBANK'
 'POS BLINKIT' 'IMPS-RENT-LANDLORD-75500265' 'ZOMATO MEDIA P L'
 'UPI-ANKIT-6430@OKAXIS' 'UPI-OLACABS@HDFCBANK' 'UPI-JIORECHARGE@PAYTM'
 'UPI-CCD@HDFCBANK' 'Swiggy*Order' 'INSTAMART BANGALORE'
 'UPI-ZOMATO-LIMITED@PAYTM' 'AVENUE SUPERMARTS' 'POS HP PETROL STATION'
 'UPI-VIKAS-6060@OKAXIS' 'POS BANGALORE RESTAURANT'
 'UPI-ZERODHA-COIN@AXIS' 'BHIM SWIGGY' 'UPI-BOOKMYSHOW@HDFCBANK'
 'BLINKIT BAN

In [14]:
vendor_keywords = {
    "Swiggy": ["SWIGGY", "BUNDL", "INSTAMART"],
    "Zomato": ["ZOMATO"],
    "Ola": ["OLA", "ANI TECHNOLOGIES"],
    "Rapido": ["RAPIDO", "ROPPEN TRANSPORTATION"],
    "BMTC": ["BMTC", "TUMMOC"],
    "Empire Restaurant": ["EMPIRE RESTAURANT"],

    "Blinkit": ["BLINKIT", "GROFERS"],
    "Zepto": ["ZEPTO"],
    "Amazon": ["AMAZON", "AMZN"],
    "Flipkart": ["FLIPKART", "FKART"],
    "Myntra": ["MYNTRA"],
    "Nykaa": ["NYKAA"],
    "BigBasket": ["BIGBASKET"],
    "DMart": ["DMART"],
    "Uber": ["UBER"],
    "Starbucks": ["STARBUCKS"],
    "Third Wave Coffee": ["THIRD WAVE", "THIRDWAVE"],
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "BookMyShow": ["BOOKMYSHOW"],
    "Zerodha": ["ZERODHA"],
    "Groww": ["GROWW"],
    "Airtel": ["AIRTEL"],
    "Jio": ["JIO"],
    "BPCL": ["BPCL"],
    "Indian Oil": ["INDIAN OIL"],
    "HP Petrol": ["HP PETROL"]
}

In [15]:
def extract_vendor(description):

    text = str(description).upper()


    if text.startswith("ATM-WDL"):
        return "Cash Withdrawal"


    if "RESTAURANT" in text:
        return "Restaurant"


    for vendor, keywords in vendor_keywords.items():

        for keyword in keywords:

            if keyword in text:
                return vendor


    if text.startswith("UPI-"):
        return "P2P Transfer"

    return "Uncategorised"

In [16]:
df["vendor_clean"] = df["Description"].apply(extract_vendor)

In [17]:
print(df["vendor_clean"].value_counts())

vendor_clean
Swiggy               219
Uncategorised        142
Zomato               121
Ola                   87
Amazon                86
Uber                  71
Restaurant            66
Zepto                 58
Rapido                55
Blinkit               55
Flipkart              47
P2P Transfer          47
Starbucks             42
BMTC                  37
Third Wave Coffee     22
Myntra                20
Cash Withdrawal       17
Nykaa                 16
Zerodha               14
DMart                 12
BigBasket             11
Indian Oil            11
Netflix               10
Groww                  9
HP Petrol              9
Spotify                8
Airtel                 6
Jio                    6
BookMyShow             4
BPCL                   2
Name: count, dtype: int64


In [18]:
uncategorised = df[df["vendor_clean"] == "Uncategorised"]

print("Uncategorised transactions:", len(uncategorised))
print("\nDescriptions still needing mapping:")

print(uncategorised["Description"].value_counts())

Uncategorised transactions: 142

Descriptions still needing mapping:
Description
KIRANAKART TECH                     13
POS MEGHANA FOODS                   12
COFFEE DAY GLOBAL                   11
AVENUE SUPERMARTS                   10
POS DINEOUT                         10
POS TRUFFLES                         9
TWC INDIA                            9
INNOVATIVE RETAIL                    8
BMS MOVIE TICKETS                    7
NEFT-TECHCRUSH LABS-SALARY MAY24     6
DISNEY HOTSTAR                       6
POS CAFE COFFEE DAY                  6
BANGALORE ELEC SUPPLY                6
BESCOM ELEC BILL                     5
BWSSB WATER BILL                     5
VI POSTPAID                          3
STAR INDIA PVT LTD                   3
FSN E-COMMERCE                       3
VODAFONE IDEA LTD                    2
BIGTREE ENTERTAINMENT                2
IMPS-RENT-LANDLORD-75500265          1
IMPS-RENT-LANDLORD-39598076          1
IMPS-RENT-LANDLORD-51148088          1
IMPS-RENT-LANDLORD-368

In [19]:
print(uncategorised["Description"].value_counts())

Description
KIRANAKART TECH                     13
POS MEGHANA FOODS                   12
COFFEE DAY GLOBAL                   11
AVENUE SUPERMARTS                   10
POS DINEOUT                         10
POS TRUFFLES                         9
TWC INDIA                            9
INNOVATIVE RETAIL                    8
BMS MOVIE TICKETS                    7
NEFT-TECHCRUSH LABS-SALARY MAY24     6
DISNEY HOTSTAR                       6
POS CAFE COFFEE DAY                  6
BANGALORE ELEC SUPPLY                6
BESCOM ELEC BILL                     5
BWSSB WATER BILL                     5
VI POSTPAID                          3
STAR INDIA PVT LTD                   3
FSN E-COMMERCE                       3
VODAFONE IDEA LTD                    2
BIGTREE ENTERTAINMENT                2
IMPS-RENT-LANDLORD-75500265          1
IMPS-RENT-LANDLORD-39598076          1
IMPS-RENT-LANDLORD-51148088          1
IMPS-RENT-LANDLORD-36852906          1
IMPS-RENT-LANDLORD-49966195          1
IMPS-RENT-LAN

In [20]:
print(
    df[df["vendor_clean"] == "Uncategorised"]["Description"]
    .value_counts()
)

Description
KIRANAKART TECH                     13
POS MEGHANA FOODS                   12
COFFEE DAY GLOBAL                   11
AVENUE SUPERMARTS                   10
POS DINEOUT                         10
POS TRUFFLES                         9
TWC INDIA                            9
INNOVATIVE RETAIL                    8
BMS MOVIE TICKETS                    7
NEFT-TECHCRUSH LABS-SALARY MAY24     6
DISNEY HOTSTAR                       6
POS CAFE COFFEE DAY                  6
BANGALORE ELEC SUPPLY                6
BESCOM ELEC BILL                     5
BWSSB WATER BILL                     5
VI POSTPAID                          3
STAR INDIA PVT LTD                   3
FSN E-COMMERCE                       3
VODAFONE IDEA LTD                    2
BIGTREE ENTERTAINMENT                2
IMPS-RENT-LANDLORD-75500265          1
IMPS-RENT-LANDLORD-39598076          1
IMPS-RENT-LANDLORD-51148088          1
IMPS-RENT-LANDLORD-36852906          1
IMPS-RENT-LANDLORD-49966195          1
IMPS-RENT-LAN

In [21]:
category_map = {
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Blinkit": "Quick Commerce",
    "Zepto": "Quick Commerce",
    "BigBasket": "Groceries",

    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",

    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",

    "Restaurant": "Restaurants",
    "Empire Restaurant": "Restaurants",

    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",

    "Zerodha": "Investments",
    "Groww": "Investments",

    "Airtel": "Utilities",
    "Jio": "Utilities",

    "DMart": "Groceries",

    "BPCL": "Fuel",
    "Indian Oil": "Fuel",
    "HP Petrol": "Fuel",

    "BookMyShow": "Entertainment",

    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal"
}

In [22]:
df["category"] = df["vendor_clean"].map(category_map)

In [23]:
print(df[["Description", "vendor_clean", "category"]].head(20))

                         Description       vendor_clean           category
0                 AMAZON SELLER SVCS             Amazon         E-commerce
1                          BHIM-BMTC               BMTC          Transport
2   NEFT-TECHCRUSH LABS-SALARY MAY24      Uncategorised                NaN
3               UPI-AMAN-8934@OKAXIS       P2P Transfer  Personal Transfer
4                       BHIM-BLINKIT            Blinkit     Quick Commerce
5                         BHIM ZEPTO              Zepto     Quick Commerce
6             UPI-UBER-2426@HDFCBANK               Uber          Transport
7                       BHIM-BLINKIT            Blinkit     Quick Commerce
8               POS SWIGGY BANGALORE             Swiggy      Food Delivery
9              UPI-GROWWPAY@HDFCBANK              Groww        Investments
10                      OLA ELECTRIC                Ola          Transport
11                 BMS MOVIE TICKETS      Uncategorised                NaN
12                     PO

In [24]:
print("Missing categories:", df["category"].isna().sum())

Missing categories: 142


In [25]:
print(
    df[df["category"].isna()][
        ["Description", "vendor_clean"]
    ].value_counts()
)

Description                       vendor_clean 
KIRANAKART TECH                   Uncategorised    13
POS MEGHANA FOODS                 Uncategorised    12
COFFEE DAY GLOBAL                 Uncategorised    11
POS DINEOUT                       Uncategorised    10
AVENUE SUPERMARTS                 Uncategorised    10
POS TRUFFLES                      Uncategorised     9
TWC INDIA                         Uncategorised     9
INNOVATIVE RETAIL                 Uncategorised     8
BMS MOVIE TICKETS                 Uncategorised     7
BANGALORE ELEC SUPPLY             Uncategorised     6
NEFT-TECHCRUSH LABS-SALARY MAY24  Uncategorised     6
DISNEY HOTSTAR                    Uncategorised     6
POS CAFE COFFEE DAY               Uncategorised     6
BESCOM ELEC BILL                  Uncategorised     5
BWSSB WATER BILL                  Uncategorised     5
VI POSTPAID                       Uncategorised     3
FSN E-COMMERCE                    Uncategorised     3
STAR INDIA PVT LTD                

In [26]:
print(df["category"].value_counts())

category
Food Delivery        340
Transport            250
E-commerce           169
Quick Commerce       113
Restaurants           66
Cafe                  64
Personal Transfer     47
Investments           23
Groceries             23
Fuel                  22
Subscriptions         18
Cash Withdrawal       17
Utilities             12
Entertainment          4
Name: count, dtype: int64


In [27]:
category_spend = (
    df[df["type"] == "debit"]
    .groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
)

print(category_spend)

category
E-commerce           599581.0
Investments          248160.0
Food Delivery        149366.0
Fuel                  74255.0
Restaurants           69199.0
Quick Commerce        60152.0
Transport             57474.0
Personal Transfer     56078.0
Cash Withdrawal       45500.0
Groceries             33398.0
Cafe                  22100.0
Utilities              6097.0
Subscriptions          5504.0
Entertainment          2692.0
Name: amount, dtype: float64


In [28]:
total_debits = df.loc[
    df["type"] == "debit", "amount"
].sum()

category_percentage = (
    category_spend / total_debits * 100
)

print(category_percentage.round(2))

category
E-commerce           35.71
Investments          14.78
Food Delivery         8.90
Fuel                  4.42
Restaurants           4.12
Quick Commerce        3.58
Transport             3.42
Personal Transfer     3.34
Cash Withdrawal       2.71
Groceries             1.99
Cafe                  1.32
Utilities             0.36
Subscriptions         0.33
Entertainment         0.16
Name: amount, dtype: float64


In [29]:
print("Missing categories:", df["category"].isna().sum())

Missing categories: 142


In [30]:
print(df["category"].value_counts())

category
Food Delivery        340
Transport            250
E-commerce           169
Quick Commerce       113
Restaurants           66
Cafe                  64
Personal Transfer     47
Investments           23
Groceries             23
Fuel                  22
Subscriptions         18
Cash Withdrawal       17
Utilities             12
Entertainment          4
Name: count, dtype: int64


In [31]:
print(category_spend)

category
E-commerce           599581.0
Investments          248160.0
Food Delivery        149366.0
Fuel                  74255.0
Restaurants           69199.0
Quick Commerce        60152.0
Transport             57474.0
Personal Transfer     56078.0
Cash Withdrawal       45500.0
Groceries             33398.0
Cafe                  22100.0
Utilities              6097.0
Subscriptions          5504.0
Entertainment          2692.0
Name: amount, dtype: float64


In [32]:
missing_categories = df[df["category"].isna()]

print(missing_categories["Description"].value_counts())

Description
KIRANAKART TECH                     13
POS MEGHANA FOODS                   12
COFFEE DAY GLOBAL                   11
AVENUE SUPERMARTS                   10
POS DINEOUT                         10
POS TRUFFLES                         9
TWC INDIA                            9
INNOVATIVE RETAIL                    8
BMS MOVIE TICKETS                    7
NEFT-TECHCRUSH LABS-SALARY MAY24     6
DISNEY HOTSTAR                       6
POS CAFE COFFEE DAY                  6
BANGALORE ELEC SUPPLY                6
BESCOM ELEC BILL                     5
BWSSB WATER BILL                     5
VI POSTPAID                          3
STAR INDIA PVT LTD                   3
FSN E-COMMERCE                       3
VODAFONE IDEA LTD                    2
BIGTREE ENTERTAINMENT                2
IMPS-RENT-LANDLORD-75500265          1
IMPS-RENT-LANDLORD-39598076          1
IMPS-RENT-LANDLORD-51148088          1
IMPS-RENT-LANDLORD-36852906          1
IMPS-RENT-LANDLORD-49966195          1
IMPS-RENT-LAN

In [34]:
print(missing_categories[["Description","vendor_clean"]].value_counts())

Description                       vendor_clean 
KIRANAKART TECH                   Uncategorised    13
POS MEGHANA FOODS                 Uncategorised    12
COFFEE DAY GLOBAL                 Uncategorised    11
POS DINEOUT                       Uncategorised    10
AVENUE SUPERMARTS                 Uncategorised    10
POS TRUFFLES                      Uncategorised     9
TWC INDIA                         Uncategorised     9
INNOVATIVE RETAIL                 Uncategorised     8
BMS MOVIE TICKETS                 Uncategorised     7
BANGALORE ELEC SUPPLY             Uncategorised     6
NEFT-TECHCRUSH LABS-SALARY MAY24  Uncategorised     6
DISNEY HOTSTAR                    Uncategorised     6
POS CAFE COFFEE DAY               Uncategorised     6
BESCOM ELEC BILL                  Uncategorised     5
BWSSB WATER BILL                  Uncategorised     5
VI POSTPAID                       Uncategorised     3
FSN E-COMMERCE                    Uncategorised     3
STAR INDIA PVT LTD                

In [35]:
df["category"] = df["vendor_clean"].map(category_map)

In [37]:
credits = df[df["type"]=="credit"]
debits = df[df["type"]=="debit"]

print("Credit transactions: ",len(credits))
print("Debit transactions: ",len(debits))

Credit transactions:  6
Debit transactions:  1304


In [38]:
total_credits = credits["amount"].sum()
total_debits = debits["amount"].sum()

print("Total credits: ",total_credits)
print("Total debits: ",total_debits)

Total credits:  509774.0
Total debits:  1678901.0


In [39]:
net_savings = total_credits - total_debits

print("Net Savings:", net_savings)

Net Savings: -1169127.0


In [40]:
savings_rate = (net_savings / total_credits) * 100

print("Savings Rate:", round(savings_rate, 2), "%")

Savings Rate: -229.34 %


In [41]:
print("=" * 50)
print("SPENDING OVERVIEW")
print("=" * 50)

print(f"Total Credits : ₹{total_credits:,.2f}")
print(f"Total Debits  : ₹{total_debits:,.2f}")
print(f"Net Savings   : ₹{net_savings:,.2f}")
print(f"Savings Rate  : {savings_rate:.2f}%")

SPENDING OVERVIEW
Total Credits : ₹509,774.00
Total Debits  : ₹1,678,901.00
Net Savings   : ₹-1,169,127.00
Savings Rate  : -229.34%


In [42]:
category_spend = (debits.groupby("category")["amount"].sum().sort_values(ascending=False))

print(category_spend)

category
E-commerce           599581.0
Investments          248160.0
Food Delivery        149366.0
Fuel                  74255.0
Restaurants           69199.0
Quick Commerce        60152.0
Transport             57474.0
Personal Transfer     56078.0
Cash Withdrawal       45500.0
Groceries             33398.0
Cafe                  22100.0
Utilities              6097.0
Subscriptions          5504.0
Entertainment          2692.0
Name: amount, dtype: float64


In [44]:
category_summary = pd.DataFrame({
    "spending": category_spend,
    "percentage": category_percentage
})

print(category_summary.round(2))

                   spending  percentage
category                               
E-commerce         599581.0       35.71
Investments        248160.0       14.78
Food Delivery      149366.0        8.90
Fuel                74255.0        4.42
Restaurants         69199.0        4.12
Quick Commerce      60152.0        3.58
Transport           57474.0        3.42
Personal Transfer   56078.0        3.34
Cash Withdrawal     45500.0        2.71
Groceries           33398.0        1.99
Cafe                22100.0        1.32
Utilities            6097.0        0.36
Subscriptions        5504.0        0.33
Entertainment        2692.0        0.16


In [45]:
vendor_spend = (
    debits
    .groupby("vendor_clean")["amount"]
    .sum()
    .sort_values(ascending=False)
)

print(vendor_spend.head(5))

vendor_clean
Amazon           328530.0
Uncategorised    249345.0
Zerodha          210000.0
Flipkart         177510.0
Swiggy            94050.0
Name: amount, dtype: float64


In [46]:
top_vendors = pd.DataFrame({
    "vendor": vendor_spend.head(5).index,
    "spending": vendor_spend.head(5).values
})

print(top_vendors)

          vendor  spending
0         Amazon  328530.0
1  Uncategorised  249345.0
2        Zerodha  210000.0
3       Flipkart  177510.0
4         Swiggy   94050.0


In [47]:
top_categories = pd.DataFrame({
    "category": category_spend.head(5).index,
    "spending": category_spend.head(5).values
})

print(top_categories)

        category  spending
0     E-commerce  599581.0
1    Investments  248160.0
2  Food Delivery  149366.0
3           Fuel   74255.0
4    Restaurants   69199.0


In [48]:
print("=" * 60)
print("             FEATURE 4 - SPENDING OVERVIEW")
print("=" * 60)

print("\nFINANCIAL SUMMARY")
print("-" * 60)
print(f"Total Credits : ₹{total_credits:,.2f}")
print(f"Total Debits  : ₹{total_debits:,.2f}")
print(f"Net Savings   : ₹{net_savings:,.2f}")
print(f"Savings Rate  : {savings_rate:.2f}%")

print("\nTOP 5 CATEGORIES")
print("-" * 60)

for category, amount in category_spend.head(5).items():
    percentage = (amount / total_debits) * 100
    print(f"{category:<20} ₹{amount:>12,.2f}  ({percentage:.2f}%)")

print("\nTOP 5 VENDORS")
print("-" * 60)

for vendor, amount in vendor_spend.head(5).items():
    print(f"{vendor:<25} ₹{amount:>12,.2f}")

print("=" * 60)

             FEATURE 4 - SPENDING OVERVIEW

FINANCIAL SUMMARY
------------------------------------------------------------
Total Credits : ₹509,774.00
Total Debits  : ₹1,678,901.00
Net Savings   : ₹-1,169,127.00
Savings Rate  : -229.34%

TOP 5 CATEGORIES
------------------------------------------------------------
E-commerce           ₹  599,581.00  (35.71%)
Investments          ₹  248,160.00  (14.78%)
Food Delivery        ₹  149,366.00  (8.90%)
Fuel                 ₹   74,255.00  (4.42%)
Restaurants          ₹   69,199.00  (4.12%)

TOP 5 VENDORS
------------------------------------------------------------
Amazon                    ₹  328,530.00
Uncategorised             ₹  249,345.00
Zerodha                   ₹  210,000.00
Flipkart                  ₹  177,510.00
Swiggy                    ₹   94,050.00


In [49]:
df["month"] = df["date"].dt.to_period("M")

In [50]:
print(df["month"].value_counts().sort_index())

month
2024-01    219
2024-02    213
2024-03    217
2024-04    222
2024-05    224
2024-06    215
Freq: M, Name: count, dtype: int64


In [52]:
monthly_spend = (df[df["type"] == "debit"].groupby("month")["amount"].sum())

print(monthly_spend)

month
2024-01    290767.0
2024-02    234071.0
2024-03    330458.0
2024-04    251382.0
2024-05    277306.0
2024-06    294917.0
Freq: M, Name: amount, dtype: float64


In [53]:
monthly_category = pd.pivot_table(
    df[df["type"] == "debit"],
    values="amount",
    index="category",
    columns="month",
    aggfunc="sum",
    fill_value=0
)

print(monthly_category)

month              2024-01  2024-02   2024-03  2024-04  2024-05   2024-06
category                                                                 
Cafe                1895.0   2965.0    3983.0   4772.0   3994.0    4491.0
Cash Withdrawal     2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce         98623.0  92738.0  105977.0  69219.0  95776.0  137248.0
Entertainment        952.0      0.0     562.0   1178.0      0.0       0.0
Food Delivery      20861.0  25272.0   25166.0  29237.0  23950.0   24880.0
Fuel               23058.0   2079.0   21035.0  18013.0   7188.0    2882.0
Groceries          11763.0   7517.0    1865.0   4916.0   5888.0    1449.0
Investments        38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Personal Transfer  18260.0   6758.0   12955.0   4925.0   6869.0    6311.0
Quick Commerce      9995.0  12459.0   12229.0   8221.0  10139.0    7109.0
Restaurants        12624.0  16432.0    6443.0   6118.0  14382.0   13200.0
Subscriptions        453.0   1036.0   

In [54]:
first_month = monthly_category.columns.min()
last_month = monthly_category.columns.max()

print("First month:", first_month)
print("Last month:", last_month)

First month: 2024-01
Last month: 2024-06


In [55]:
category_change = pd.DataFrame({
    "first_month": monthly_category[first_month],
    "last_month": monthly_category[last_month]
})

category_change["change"] = (category_change["last_month"]- category_change["first_month"])

print(category_change.sort_values("change", ascending=False))

                   first_month  last_month   change
category                                           
E-commerce             98623.0    137248.0  38625.0
Cash Withdrawal         2000.0     17000.0  15000.0
Food Delivery          20861.0     24880.0   4019.0
Cafe                    1895.0      4491.0   2596.0
Subscriptions            453.0      1341.0    888.0
Restaurants            12624.0     13200.0    576.0
Utilities               1137.0       729.0   -408.0
Entertainment            952.0         0.0   -952.0
Quick Commerce          9995.0      7109.0  -2886.0
Transport              11005.0      6996.0  -4009.0
Groceries              11763.0      1449.0 -10314.0
Personal Transfer      18260.0      6311.0 -11949.0
Investments            38432.0     23330.0 -15102.0
Fuel                   23058.0      2882.0 -20176.0


In [56]:
category_change["percentage_change"] = (category_change["change"]/ category_change["first_month"].replace(0, np.nan)) * 100

print(category_change.round(2))

                   first_month  last_month   change  percentage_change
category                                                              
Cafe                    1895.0      4491.0   2596.0             136.99
Cash Withdrawal         2000.0     17000.0  15000.0             750.00
E-commerce             98623.0    137248.0  38625.0              39.16
Entertainment            952.0         0.0   -952.0            -100.00
Food Delivery          20861.0     24880.0   4019.0              19.27
Fuel                   23058.0      2882.0 -20176.0             -87.50
Groceries              11763.0      1449.0 -10314.0             -87.68
Investments            38432.0     23330.0 -15102.0             -39.30
Personal Transfer      18260.0      6311.0 -11949.0             -65.44
Quick Commerce          9995.0      7109.0  -2886.0             -28.87
Restaurants            12624.0     13200.0    576.0               4.56
Subscriptions            453.0      1341.0    888.0             196.03
Transp

In [58]:
increased = category_change[category_change["change"] > 0].sort_values("change",ascending=False)

print("Categories that increased:")
print(increased)

Categories that increased:
                 first_month  last_month   change  percentage_change
category                                                            
E-commerce           98623.0    137248.0  38625.0          39.164292
Cash Withdrawal       2000.0     17000.0  15000.0         750.000000
Food Delivery        20861.0     24880.0   4019.0          19.265615
Cafe                  1895.0      4491.0   2596.0         136.992084
Subscriptions          453.0      1341.0    888.0         196.026490
Restaurants          12624.0     13200.0    576.0           4.562738


In [59]:
decreased = category_change[category_change["change"] < 0].sort_values("change")

print("Categories that decreased:")
print(decreased)

Categories that decreased:
                   first_month  last_month   change  percentage_change
category                                                              
Fuel                   23058.0      2882.0 -20176.0         -87.501084
Investments            38432.0     23330.0 -15102.0         -39.295379
Personal Transfer      18260.0      6311.0 -11949.0         -65.438116
Groceries              11763.0      1449.0 -10314.0         -87.681714
Transport              11005.0      6996.0  -4009.0         -36.428896
Quick Commerce          9995.0      7109.0  -2886.0         -28.874437
Entertainment            952.0         0.0   -952.0        -100.000000
Utilities               1137.0       729.0   -408.0         -35.883905


In [60]:
category_percentage = (category_spend / total_debits * 100)

print(category_percentage.round(2))

category
E-commerce           35.71
Investments          14.78
Food Delivery         8.90
Fuel                  4.42
Restaurants           4.12
Quick Commerce        3.58
Transport             3.42
Personal Transfer     3.34
Cash Withdrawal       2.71
Groceries             1.99
Cafe                  1.32
Utilities             0.36
Subscriptions         0.33
Entertainment         0.16
Name: amount, dtype: float64


In [61]:
food_pct = category_percentage.get("Food Delivery", 0)

shop_pct = category_percentage.get("E-commerce", 0)

transport_pct = category_percentage.get("Transport", 0)

investment_pct = category_percentage.get("Investments", 0)

In [62]:
print("Food Delivery:", round(food_pct, 2), "%")
print("E-commerce:", round(shop_pct, 2), "%")
print("Transport:", round(transport_pct, 2), "%")
print("Investments:", round(investment_pct, 2), "%")

Food Delivery: 8.9 %
E-commerce: 35.71 %
Transport: 3.42 %
Investments: 14.78 %


In [63]:
def get_archetype(food_pct, shop_pct, transport_pct, investment_pct):

    if food_pct > 15:
        return "Foodie"

    elif shop_pct > 25:
        return "Shopper"

    elif transport_pct > 20:
        return "Commuter"

    elif investment_pct > 15:
        return "Investor"

    else:
        return "Balanced"

In [64]:
archetype = get_archetype(
    food_pct,
    shop_pct,
    transport_pct,
    investment_pct
)

print("Spending Archetype:", archetype)

Spending Archetype: Shopper


In [65]:
archetype_explanation = {
    "Foodie": "A significant share of spending goes toward food delivery.",
    "Shopper": "A significant share of spending goes toward e-commerce.",
    "Commuter": "A significant share of spending goes toward transportation.",
    "Investor": "A significant share of spending goes toward investments.",
    "Balanced": "No single spending category crosses the defined archetype thresholds."
}

print(archetype_explanation[archetype])

A significant share of spending goes toward e-commerce.


In [66]:
print("=" * 50)
print("FEATURE 6 - SPENDING ARCHETYPE")
print("=" * 50)

print(f"Food Delivery : {food_pct:.2f}%")
print(f"E-commerce    : {shop_pct:.2f}%")
print(f"Transport     : {transport_pct:.2f}%")
print(f"Investments   : {investment_pct:.2f}%")

print(f"\nArchetype: {archetype}")
print(archetype_explanation[archetype])

FEATURE 6 - SPENDING ARCHETYPE
Food Delivery : 8.90%
E-commerce    : 35.71%
Transport     : 3.42%
Investments   : 14.78%

Archetype: Shopper
A significant share of spending goes toward e-commerce.


In [68]:
top_category = category_spend.idxmax()
top_category_amount = category_spend.max()

top_category_pct = (top_category_amount / total_debits) * 100

print(
    f"Top spending category: {top_category} "
    f"with ₹{top_category_amount:,.2f} "
    f"({top_category_pct:.2f}% of debit spending)")

Top spending category: E-commerce with ₹599,581.00 (35.71% of debit spending)


In [69]:
top_vendor = vendor_spend.idxmax()
top_vendor_amount = vendor_spend.max()

print(
    f"Top vendor: {top_vendor} "
    f"with ₹{top_vendor_amount:,.2f}")

Top vendor: Amazon with ₹328,530.00


In [70]:
highest_month = monthly_spend.idxmax()
highest_month_amount = monthly_spend.max()

print(
    f"Highest spending month: {highest_month} "
    f"with ₹{highest_month_amount:,.2f}")

Highest spending month: 2024-03 with ₹330,458.00


In [71]:
lowest_month = monthly_spend.idxmin()
lowest_month_amount = monthly_spend.min()

print(
    f"Lowest spending month: {lowest_month} "
    f"with ₹{lowest_month_amount:,.2f}")

Lowest spending month: 2024-02 with ₹234,071.00


In [73]:
fastest_growing_category = category_change["change"].idxmax()
fastest_growth = category_change.loc[
    fastest_growing_category, "change"]

print(
    f"Category with largest increase: "
    f"{fastest_growing_category} "
    f"({fastest_growth:+,.2f})")

Category with largest increase: E-commerce (+38,625.00)


In [74]:
insights = []

insights.append(
    f"{top_category} was the highest spending category, "
    f"accounting for ₹{top_category_amount:,.2f} "
    f"({top_category_pct:.2f}% of debit spending).")

insights.append(
    f"{top_vendor} was the highest-spending vendor, "
    f"with total spending of ₹{top_vendor_amount:,.2f}.")

insights.append(
    f"{highest_month} recorded the highest total spending "
    f"at ₹{highest_month_amount:,.2f}.")

insights.append(
    f"{lowest_month} recorded the lowest total spending "
    f"at ₹{lowest_month_amount:,.2f}.")

insights.append(
    f"{fastest_growing_category} showed the largest increase "
    f"between the first and last month, changing by "
    f"₹{fastest_growth:,.2f}.")

In [75]:
print("=" * 60)
print("FEATURE 7 - SMART INSIGHTS")
print("=" * 60)

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

FEATURE 7 - SMART INSIGHTS
1. E-commerce was the highest spending category, accounting for ₹599,581.00 (35.71% of debit spending).
2. Amazon was the highest-spending vendor, with total spending of ₹328,530.00.
3. 2024-03 recorded the highest total spending at ₹330,458.00.
4. 2024-02 recorded the lowest total spending at ₹234,071.00.
5. E-commerce showed the largest increase between the first and last month, changing by ₹38,625.00.


In [76]:
largest_transaction = debits.loc[
    debits["amount"].idxmax()]

print(
    "Largest transaction:",
    largest_transaction["Description"],
    "₹" + f"{largest_transaction['amount']:,.2f}")

Largest transaction: AMAZONIN MARKETPLACE ₹22,008.00


In [77]:
insights.append(
    f"The largest individual debit transaction was "
    f"{largest_transaction['Description']} "
    f"for ₹{largest_transaction['amount']:,.2f}.")

In [80]:
print("=" * 60)
print("CATEGORY SPENDING")
print("=" * 60)

for category, amount in category_spend.head(10).items():

    bar_length = int((amount / category_spend.max()) * 30)

    print(
        f"{category:<20} "
        f"{'#' * bar_length} "
        f"₹{amount:,.0f}")

CATEGORY SPENDING
E-commerce           ############################## ₹599,581
Investments          ############ ₹248,160
Food Delivery        ####### ₹149,366
Fuel                 ### ₹74,255
Restaurants          ### ₹69,199
Quick Commerce       ### ₹60,152
Transport            ## ₹57,474
Personal Transfer    ## ₹56,078
Cash Withdrawal      ## ₹45,500
Groceries            # ₹33,398


In [79]:
print("\n" + "=" * 60)
print("MONTHLY SPENDING")
print("=" * 60)

max_monthly = monthly_spend.max()

for month, amount in monthly_spend.items():

    bar_length = int(
        (amount / max_monthly) * 30)

    print(
        f"{str(month):<10} "
        f"{'#' * bar_length} "
        f"₹{amount:,.0f}")


MONTHLY SPENDING
2024-01    ########################## ₹290,767
2024-02    ##################### ₹234,071
2024-03    ############################## ₹330,458
2024-04    ###################### ₹251,382
2024-05    ######################### ₹277,306
2024-06    ########################## ₹294,917


In [81]:
print("\n" + "=" * 60)
print("TOP 10 VENDORS")
print("=" * 60)

for vendor, amount in vendor_spend.head(10).items():

    print(
        f"{vendor:<25} "
        f"₹{amount:,.2f}")


TOP 10 VENDORS
Amazon                    ₹328,530.00
Uncategorised             ₹249,345.00
Zerodha                   ₹210,000.00
Flipkart                  ₹177,510.00
Swiggy                    ₹94,050.00
Myntra                    ₹69,529.00
Restaurant                ₹69,199.00
P2P Transfer              ₹56,078.00
Zomato                    ₹55,316.00
Cash Withdrawal           ₹45,500.00


In [82]:
print("\n" + "=" * 70)
print("                    SpendDNA")
print("              EXECUTIVE SUMMARY")
print("=" * 70)

print("\nFINANCIAL OVERVIEW")
print("-" * 70)

print(f"Total Transactions : {len(df):,}")
print(f"Total Credits     : ₹{total_credits:,.2f}")
print(f"Total Debits      : ₹{total_debits:,.2f}")
print(f"Net Savings       : ₹{net_savings:,.2f}")
print(f"Savings Rate      : {savings_rate:.2f}%")

print("\nSPENDING PATTERN")
print("-" * 70)

print(
    f"Top Category      : {top_category} "
    f"(₹{top_category_amount:,.2f})")

print(
    f"Top Vendor        : {top_vendor} "
    f"(₹{top_vendor_amount:,.2f})")

print(
    f"Highest Spend Month: {highest_month} "
    f"(₹{highest_month_amount:,.2f})")

print(
    f"Lowest Spend Month : {lowest_month} "
    f"(₹{lowest_month_amount:,.2f})")

print(
    f"Spending Archetype : {archetype}")

print("\nKEY INSIGHTS")
print("-" * 70)

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

print("\n" + "=" * 70)


                    SpendDNA
              EXECUTIVE SUMMARY

FINANCIAL OVERVIEW
----------------------------------------------------------------------
Total Transactions : 1,310
Total Credits     : ₹509,774.00
Total Debits      : ₹1,678,901.00
Net Savings       : ₹-1,169,127.00
Savings Rate      : -229.34%

SPENDING PATTERN
----------------------------------------------------------------------
Top Category      : E-commerce (₹599,581.00)
Top Vendor        : Amazon (₹328,530.00)
Highest Spend Month: 2024-03 (₹330,458.00)
Lowest Spend Month : 2024-02 (₹234,071.00)
Spending Archetype : Shopper

KEY INSIGHTS
----------------------------------------------------------------------
1. E-commerce was the highest spending category, accounting for ₹599,581.00 (35.71% of debit spending).
2. Amazon was the highest-spending vendor, with total spending of ₹328,530.00.
3. 2024-03 recorded the highest total spending at ₹330,458.00.
4. 2024-02 recorded the lowest total spending at ₹234,071.00.
5. E-co

In [83]:
df["hour"] = df["Time"].str[:2].astype(int)

In [85]:
hourly_spending = (df[df["type"] == "debit"].groupby("hour")["amount"].sum().sort_values(ascending=False))

print(hourly_spending)

hour
10    140526.0
20    138089.0
11    123389.0
14    121010.0
13    113819.0
18    101343.0
16    100597.0
9      86701.0
21     81545.0
12     77096.0
19     67848.0
17     56604.0
22     55867.0
6      55506.0
23     48403.0
4      43901.0
8      41399.0
15     39647.0
1      38882.0
0      37969.0
3      34868.0
5      33671.0
2      20283.0
7      19938.0
Name: amount, dtype: float64


In [86]:
food_delivery = df[
    (df["category"] == "Food Delivery") &
    (df["type"] == "debit")
]

In [87]:
late_night = food_delivery[
    (food_delivery["hour"] >= 21) |
    (food_delivery["hour"] <= 2)]

late_night_percentage = (len(late_night) /len(food_delivery) *100)

print(
    "Late-night Food Delivery:",
    round(late_night_percentage, 2),
    "%")

Late-night Food Delivery: 20.88 %


In [88]:
category_mean = (df.groupby("category")["amount"].transform("mean"))

category_std = (df.groupby("category")["amount"].transform("std"))

df["z_score"] = ((df["amount"] - category_mean)/ category_std)

In [89]:
anomalies = df[
    (df["type"] == "debit") &
    (df["z_score"] > 2)
].sort_values(
    "z_score",
    ascending=False)

In [90]:
print(
    anomalies[
        [
            "date",
            "vendor_clean",
            "category",
            "amount",
            "z_score"
        ]].head(10))

           date  vendor_clean           category   amount   z_score
414  2024-02-26    Restaurant        Restaurants   8383.0  5.639213
148  2024-01-22  P2P Transfer  Personal Transfer   7264.0  5.105378
49   2024-01-07    Restaurant        Restaurants   7314.0  4.817303
1298 2024-06-26        Amazon         E-commerce  22008.0  4.054513
269  2024-02-07        Amazon         E-commerce  21986.0  4.049681
475  2024-03-05        Amazon         E-commerce  19917.0  3.595255
450  2024-03-02        Amazon         E-commerce  18273.0  3.234174
1079 2024-05-27      Flipkart         E-commerce  17831.0  3.137095
1291 2024-06-25        Amazon         E-commerce  17504.0  3.065275
1130 2024-06-03      Flipkart         E-commerce  16369.0  2.815988


In [91]:
print(df[df["category"].isna()]["vendor_clean"].value_counts())

vendor_clean
Uncategorised    142
Name: count, dtype: int64


In [92]:
print("=" * 75)
print("                         SpendDNA")
print("                  FINAL SPENDING REPORT")
print("=" * 75)

# --------------------------------------------------
# 1. BASIC DATASET SUMMARY
# --------------------------------------------------

print("\n1. DATASET SUMMARY")
print("-" * 75)

print(f"Total Transactions : {len(df):,}")
print(f"Credit Transactions: {len(credits):,}")
print(f"Debit Transactions : {len(debits):,}")

print(
    f"Date Range         : "
    f"{df['date'].min().date()} to {df['date'].max().date()}")


# --------------------------------------------------
# 2. FINANCIAL OVERVIEW
# --------------------------------------------------

print("\n2. FINANCIAL OVERVIEW")
print("-" * 75)

print(f"Total Credits      : ₹{total_credits:,.2f}")
print(f"Total Debits       : ₹{total_debits:,.2f}")
print(f"Net Savings        : ₹{net_savings:,.2f}")
print(f"Savings Rate       : {savings_rate:.2f}%")


# --------------------------------------------------
# 3. TOP SPENDING CATEGORIES
# --------------------------------------------------

print("\n3. TOP SPENDING CATEGORIES")
print("-" * 75)

for category, amount in category_spend.head(5).items():

    percentage = (amount / total_debits) * 100

    print(
        f"{category:<22}"
        f"₹{amount:>12,.2f} "
        f"({percentage:.2f}%)")


# --------------------------------------------------
# 4. TOP VENDORS
# --------------------------------------------------

print("\n4. TOP SPENDING VENDORS")
print("-" * 75)

for vendor, amount in vendor_spend.head(5).items():

    print(
        f"{vendor:<30}"
        f"₹{amount:>12,.2f}")


# --------------------------------------------------
# 5. MONTHLY SPENDING
# --------------------------------------------------

print("\n5. MONTHLY SPENDING")
print("-" * 75)

for month, amount in monthly_spend.items():

    print(
        f"{str(month):<12}"
        f"₹{amount:>12,.2f}")


highest_month = monthly_spend.idxmax()
highest_month_amount = monthly_spend.max()

lowest_month = monthly_spend.idxmin()
lowest_month_amount = monthly_spend.min()

print(
    f"\nHighest Spending Month : {highest_month} "
    f"(₹{highest_month_amount:,.2f})")

print(
    f"Lowest Spending Month  : {lowest_month} "
    f"(₹{lowest_month_amount:,.2f})")


# --------------------------------------------------
# 6. SPENDING ARCHETYPE
# --------------------------------------------------

print("\n6. SPENDING ARCHETYPE")
print("-" * 75)

print(f"Food Delivery : {food_pct:.2f}%")
print(f"E-commerce    : {shop_pct:.2f}%")
print(f"Transport     : {transport_pct:.2f}%")
print(f"Investments   : {investment_pct:.2f}%")

print(f"\nSpending Archetype: {archetype}")
print(archetype_explanation[archetype])


# --------------------------------------------------
# 7. TIME-OF-DAY ANALYSIS
# --------------------------------------------------

print("\n7. TIME-OF-DAY ANALYSIS")
print("-" * 75)

peak_hour = hourly_spending.idxmax()
peak_hour_amount = hourly_spending.max()

print(
    f"Peak Spending Hour : {peak_hour}:00")

print(
    f"Spending at Peak   : ₹{peak_hour_amount:,.2f}")

print(
    f"Late-night Food Delivery: "
    f"{late_night_percentage:.2f}%")


# --------------------------------------------------
# 8. ANOMALY ANALYSIS
# --------------------------------------------------

print("\n8. ANOMALY ANALYSIS")
print("-" * 75)

print(
    f"Unusual Transactions (z > 2): "
    f"{len(anomalies)}")

if len(anomalies) > 0:

    print("\nTop Unusual Transactions:")

    print(
        anomalies[
            [
                "date",
                "vendor_clean",
                "category",
                "amount",
                "z_score"
            ]
        ].head(5).to_string(index=False))


# --------------------------------------------------
# 9. SMART INSIGHTS
# --------------------------------------------------

print("\n9. KEY INSIGHTS")
print("-" * 75)

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")


# --------------------------------------------------
# 10. DATA QUALITY CHECK
# --------------------------------------------------

print("\n10. DATA QUALITY CHECK")
print("-" * 75)

print(
    f"Missing Vendor Values    : "
    f"{df['vendor_clean'].isna().sum()}")

print(
    f"Missing Category Values  : "
    f"{df['category'].isna().sum()}")

print(
    f"Missing Amount Values    : "
    f"{df['amount'].isna().sum()}")

print(
    f"Missing Date Values      : "
    f"{df['date'].isna().sum()}")


# --------------------------------------------------
# FINAL MESSAGE
# --------------------------------------------------

print("\n" + "=" * 75)
print("                    ANALYSIS COMPLETE")
print("=" * 75)

                         SpendDNA
                  FINAL SPENDING REPORT

1. DATASET SUMMARY
---------------------------------------------------------------------------
Total Transactions : 1,310
Credit Transactions: 6
Debit Transactions : 1,304
Date Range         : 2024-01-01 to 2024-06-30

2. FINANCIAL OVERVIEW
---------------------------------------------------------------------------
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Savings        : ₹-1,169,127.00
Savings Rate       : -229.34%

3. TOP SPENDING CATEGORIES
---------------------------------------------------------------------------
E-commerce            ₹  599,581.00 (35.71%)
Investments           ₹  248,160.00 (14.78%)
Food Delivery         ₹  149,366.00 (8.90%)
Fuel                  ₹   74,255.00 (4.42%)
Restaurants           ₹   69,199.00 (4.12%)

4. TOP SPENDING VENDORS
---------------------------------------------------------------------------
Amazon                        ₹  328,530.00
Unc